In [2]:
import os
import kagglehub
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pipeline import build_pipeline

path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")
df = pd.read_csv(os.path.join(path, "AB_NYC_2019.csv"))
X = df.drop(columns=["price"])
y = df["price"]
cv = KFold(n_splits=5, shuffle=True, random_state=42)

Using Colab cache for faster access to the 'new-york-city-airbnb-open-data' dataset.


In [3]:
# R6/R7: correct pipeline stays inside cross-validation
model = build_pipeline()
scores = cross_val_score(model, X, y, cv=cv, scoring="r2", n_jobs=-1)
print("Correct pipeline CV R2:", scores)
print("Mean:", scores.mean(), "Std:", scores.std())

Correct pipeline CV R2: [0.13964052 0.15838202 0.10938932 0.11381123 0.06605714]
Mean: 0.1174560433507948 Std: 0.0312643268335297


In [4]:
# R7.1: deliberate leakage experiment
# This version fits preprocessing on all rows before CV. That lets validation
# rows influence imputation, scaling, and category grouping.
numeric_features = ["latitude", "longitude", "minimum_nights", "number_of_reviews", "reviews_per_month", "calculated_host_listings_count", "availability_365"]
categorical_features = ["neighbourhood_group", "neighbourhood", "room_type"]

bad_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ]), categorical_features),
])

X_bad = bad_preprocessor.fit_transform(X)
bad_model = Pipeline([
    ("model", TransformedTargetRegressor(regressor=Ridge(alpha=10), func=np.log1p, inverse_func=np.expm1))
])
bad_scores = cross_val_score(bad_model, X_bad, y, cv=cv, scoring="r2", n_jobs=-1)
print("Leaky CV R2:", bad_scores)
print("Mean:", bad_scores.mean(), "Std:", bad_scores.std())
print("Leaky - correct mean:", bad_scores.mean() - scores.mean())

Leaky CV R2: [0.13126296 0.14189106 0.09856653 0.10255564 0.062773  ]
Mean: 0.10740983994158138 Std: 0.02775658682732692
Leaky - correct mean: -0.01004620340921343


In [5]:
# R7.2: unseen category must be accepted by the final pipeline
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
unseen = X_test.copy()
unseen.loc[unseen.index[0], "neighbourhood"] = "NEVER_SEEN_NEIGHBOURHOOD"

good_model = build_pipeline()
good_model.fit(X_train, y_train)
print("Seen during training:", "NEVER_SEEN_NEIGHBOURHOOD" in set(X_train["neighbourhood"].dropna()))
print("Prediction with unseen category:", good_model.predict(unseen.iloc[[0]])[0])

Seen during training: False
Prediction with unseen category: 147.2154409094268


In [6]:
# Without handle_unknown="ignore", the same unseen value raises an error.
bad_encoder = OneHotEncoder(handle_unknown="error", min_frequency=20)
bad_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", bad_encoder)])
bad_preprocessor = ColumnTransformer([("cat", bad_pipe, ["neighbourhood"])])
bad_preprocessor.fit(X_train[["neighbourhood"]])

try:
    bad_preprocessor.transform(unseen.iloc[[0]][["neighbourhood"]])
    print("Unexpected: no error")
except ValueError as e:
    print("Expected ValueError:", type(e).__name__)

Expected ValueError: ValueError


In [7]:
# R6: all-NaN feature in the training data should not break the pipeline.
train_check = X_train.copy()
train_check["reviews_per_month"] = np.nan
robust_model = build_pipeline()
robust_model.fit(train_check, y_train)
print("Prediction with all-NaN training feature:", robust_model.predict(X_test.iloc[[0]])[0])

Prediction with all-NaN training feature: 119.96279189853529
